# 03 Non-streaming vs Streaming：首 token 时延与总耗时

本示例对比非流式与流式请求的首 token 时延（TTFT）与总耗时。

In [ ]:
import os
import time
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("HY3_API_KEY"),
    base_url="https://tokenhub.tencentmaas.com/v1",
)

PROMPT = "请用 300 字左右解释机器学习中的梯度下降算法。"

In [ ]:
start = time.perf_counter()
response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": PROMPT}],
    temperature=0.7,
    max_tokens=512,
    stream=False,
)
end = time.perf_counter()

print("非流式总耗时:", f"{end - start:.3f}s")
print(response.choices[0].message.content[:200], "...")

In [ ]:
start = time.perf_counter()
first_token_time = None
response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": PROMPT}],
    temperature=0.7,
    max_tokens=512,
    stream=True,
)

full_content = ""
for chunk in response:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        full_content += delta.content

end = time.perf_counter()
print(f"流式 TTFT: {first_token_time - start:.3f}s")
print(f"流式总耗时: {end - start:.3f}s")
print(full_content[:200], "...")